# Dataset Cleaning/Wrangling
Before we go about training our models on the data we usually need to clean or wrangle it. This is required in many cases as most datasets are raw data that have unneccesary columns, missing values, and incorrect type information. Today you will be focusing on cleaning the **Rain in Australia (weatherAUS)** dataset which records meteorological data as to allow you to predict whether it rained that day or the next. Through these exercises we hope you gain a good understanding on the steps required before training on a dataset can be done.

## Imports
Below are the associated imports for the project. Don't be afraid to add more if you want.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from typing import List, Dict, Tuple
import numpy as np

## Dataset
Lets have a look into what features and aspects are found within our dataset. A method we've used already is `head` however another useful one to keep note of is `info` which gives information the amount of values, their data type, and the amount of `NaN` values. Where a `NaN` value is any data that isn't available. This information will be important later when filling in our data and ensuring it has the correct types.

In [2]:
dataset = pd.read_csv("../data/weatherAUS.csv")
dataset.head() # You should have seen this already

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [10]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142193 entries, 0 to 145458
Data columns (total 24 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Location       142193 non-null  object 
 1   MinTemp        141556 non-null  float64
 2   MaxTemp        141871 non-null  float64
 3   Rainfall       140787 non-null  float64
 4   Sunshine       74377 non-null   float64
 5   WindGustDir    132863 non-null  object 
 6   WindGustSpeed  132923 non-null  float64
 7   WindDir9am     132180 non-null  object 
 8   WindDir3pm     138415 non-null  object 
 9   WindSpeed9am   140845 non-null  float64
 10  WindSpeed3pm   139563 non-null  float64
 11  Humidity9am    140419 non-null  float64
 12  Humidity3pm    138583 non-null  float64
 13  Pressure9am    128179 non-null  float64
 14  Pressure3pm    128212 non-null  float64
 15  Cloud9am       88536 non-null   float64
 16  Cloud3pm       85099 non-null   float64
 17  Temp9am        141289 non-null  fl

## **(Question 1)** Feature Extraction
Before we go about wrangling our data we first may want to choose features we believe are most important as to decrease the complexity of the training data. This usually involves removing specific columns. From the previous notebook we did this by only using one of the columns.
```python
data = pd.read_csv('../data/insurance.csv')
x = data["age"].to_numpy() # We extract only 1 feature for this data
```
In many cases its common to train on more than just one column as to improve our accuracy. The tradeoff of for more features is that it will take longer to train. This means its good to remove features that have little correlation with the result as they increase training time with little-to-no benefit.

**Remove Evaporation from the dataset**.

<details>
  <summary>Hint</summary>
    The function <code>dataset.drop([...], axis=1, inplace=True)</code> may be useful.
</details>

In [4]:
# Remove columns here
#!Answer
dataset.drop(['Evaporation'], axis=1, inplace=True)

## **(Question 2)** Dates
A type of data that is common in the majority of datasets are dates. These are represented in forms such as `d/m/y`, `m/day/y`, etc. Pandas provides you with the `Date` type for managing the fact date can be represented in multiple ways. Still not all models such as neural networks necessarily support the date type so its usually better to split them into individual columns to make them easier to train and more generalisable in the long term.

**Split the dates into Days, Months, Years and remove the old Date column**. 

<details>
  <summary>Hint</summary>
    To first convert a column to a data use `pd.to_datetime(...)`.
</details>

In [5]:
# Apply modifications to the dataset here
#!Answer
# Convert to datetime if not already
dataset['Date'] = pd.to_datetime(dataset['Date'])

# Extract day, month, year
dataset['Day'] = dataset['Date'].dt.day
dataset['Month'] = dataset['Date'].dt.month
dataset['Year'] = dataset['Date'].dt.year

# Drop the original date column
dataset.drop(['Date'], axis=1, inplace=True)

In [6]:
assert "Day" in dataset.columns or "day" in dataset.columns, "Please add a Day column"
assert "Month" in dataset.columns or "month" in dataset.columns, "Please add a Month column"
assert "Year" in dataset.columns or "year" in dataset.columns, "Please add a Year column"
assert "Date" not in dataset.columns, "Please remove Date column"

## **(Question 3)** Not Available Data
From running `info` and `head` above we may see many incomplete fields within our dataset. For example despite their being 145460 entries only 142199 have a rain today value. Our first order of business is correcting these mistakes. There are two main approaches to this:
- Removing unused data. This is at the cost of removing data that may be useful.
- Replacing unused data (possibly with a calculation such as mean or median). This is at the cost of simplifying the model's complexity and adding bias.

### (Part A) Removing NaN Values
**Remove all rows with `NaN` values in the RainTomorrow column**.

<details>
  <summary>Hint</summary>
    Some functions which may be useful depending on the approach used are:
    <ul>
        <li><code>dataset[column].dropna()</code></li>
    </ul>
</details>

In [7]:
# Place modifications to the dataset here...
#!Answer
dataset.dropna(subset=['RainTomorrow'], inplace=True)

### (Part B) Filling NaN Values
There are many different strategies for filling `NaN` values depending on the degree of simplication you are willing to apply to the dataset. If you have a low amount of `NaN` values filling with the mean, median or mode can be a good approach to keep the rows in the dataset. Another common strategy for filling `NaN` values is to use a linear regression to find the values. This is good in cases where rows show correlation.

**Fill the remaining `NaN` values with an subsitute value**. We've helped you by giving you rows that can easily be filled.

<details>
  <summary>Hint</summary>
    Some functions which may be useful depending on the approach used are:
    <ul>
        <li><code>dataset[column].fillna(...)</code></li>
    </ul>
</details>

In [27]:
numeric_columns = ['MinTemp', 'MaxTemp', 'Rainfall', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm']
#!Answer
for col in numeric_columns:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0]) # This can also be 

In [28]:
assert dataset[numeric_columns].isna().sum().sum() == 0, 'Still have NaN rows to remove' # Checking all NaN values are filled

### **(Question 4)** Encoding Arguments
Looking back at the `dataset.info()` we see that a lot of data we are reliant on has the datatype `object`rather than boolean/integer. This data is mostly string labels, such as compass directions. This will cause problems later on as to generate weights and biases we need to encode these as numeric values. This can be done through two main approaches:
- **Label encoding**, where every category is encoded as an integer. This can be done automatically or done by using a relevant number to the data, such as the amount of appearences. 
- **On hot encoding**, where each category is represented as $n$ Boolean columns, where $n$ is the amount of categories. Below is an example with `RainToday`.

| ID | RainToday |
| ---- | ----- |
| 0 | Yes |
| 1 | No |

| ID | Yes | No |
| --- | ---- | ----- |
| 0 | 1 | 0 |
| 1 | 0 | 1 |

**Encode object arguments as integer values**. A place to start is by encoding RainToday and RainTomorrow as boolean values instead of as Yes or No. After that if you have time try either doing a one hot encoding, label encoding or any other techniques you find online to encode the columns that use compass directions.

<details>
  <summary>Hint</summary>
    Some functions which may be useful depending on the approach used are:
    <ul>
        <li><code>pd.get_dummies</code></li>
        <li><code>dataset[...].map({})</code></li>
        <li><code>OneHotEncoder</code> from sklearn</li>
        <li><code>LabelEncoder</code> from sklearn</li>
    </ul>
</details>

In [ ]:
# Encode Dataset...
#!Answer
# Mapping Yes/No to Integer
for x in ("RainToday", "RainTomorrow"):
    dataset[x] = dataset[x].map({"Yes": 1, "No": 0})

# One hot encoding example
encoded_cols = pd.get_dummies(dataset[['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']])
dataset = pd.concat([dataset, encoded_cols], axis=1)

# boolean_columns = dataset.select_dtypes(include=bool).columns
# dataset[boolean_columns] = dataset[boolean_columns].astype(int)

Given that you have gone about doing the above task nothing should be printed.

In [ ]:
non_encoded_cols = dataset.select_dtypes(include=['object']).columns
assert len(non_encoded_cols) > 0, f"The following rows need to be encoded:\n{non_encoded_cols}"